# Subclass Separation Statistics (Cell + Nucleus) - vem 1uM, 48hrs Pre-Treatment

For each frame of the `vem_1uM_48hrsbefore` dataset, tests whether the 4
subclasses (dying, persisting, slow proliferator, fast proliferator) are
statistically distinguishable morphologies - not just visually separated
in a KDE plot (`01_subclass_by_frame.ipynb`), but separated in a way a
permutation test can certify.

**This operates on CAJAL's own Gromov-Wasserstein distance matrix, never
on the 2D UMAP projection of it.** UMAP optimizes a 2D layout for visual
cluster separation, not for preserving actual distances, so any
quantitative claim built on UMAP coordinates (including a statistical
test) would be measuring UMAP's layout algorithm, not the underlying
morphology. The distance matrix is the one thing actually computed from
cell shape; this notebook is the first consumer of it that is NOT a
UMAP input.

Requires `main.py` to have been run first (produces `selected_metadata.csv`,
and `{cell,nuc}_qgw_075P_dists.csv` under `output/cajal_output/`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/vem_48hr_pretreatment/notebooks/), so the analysis dir - and
# config.py - is one level up, and the repo root - where shared/ lives -
# is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import match_labels_to_metadata
from shared.permanova import permanova, permanova_pairwise, permdisp

## Method

For each (modality, frame) combination, restricted to that frame's own
cells only:

- **PERMANOVA** (Anderson 2001) - permutation test of whether subclass
  membership explains variance in the distance matrix, i.e. whether
  subclasses occupy different *locations*. Reports a pseudo-F statistic,
  an R² effect size (fraction of total distance variance explained by
  subclass - meaningful on its own, unlike the p-value at these sample
  sizes), and a permutation p-value (999 permutations).
- **PERMDISP** (Anderson 2006) - permutation test of whether subclasses
  differ in *dispersion* (spread), computed from distances to each
  subclass's own centroid in a PCoA embedding of the same distance
  matrix. This is run alongside PERMANOVA because PERMANOVA's pseudo-F
  is sensitive to dispersion differences even when centroids coincide -
  a PERMANOVA hit with no PERMDISP hit is a clean location effect; a hit
  on both is ambiguous and reported as such below, not overstated as a
  clean separation result.
- **Pairwise PERMANOVA with Benjamini-Hochberg correction** - all 6
  pairwise subclass comparisons within that (modality, frame), correcting
  for the 6 simultaneous tests (controls the false discovery rate - a
  p < 0.05 across 6 uncorrected tests has a much higher chance of a false
  positive than a single test does).

999 permutations and seed 42 are used throughout for reproducibility
(`p = 0.001` is the smallest attainable p-value at 999 permutations: it
is `(0 + 1) / (999 + 1)`, the permutation test's own floor - not evidence
of anything beyond "this did not happen in any of 999 random relabelings").

Both tests require >= 2 subclasses present in a frame; a subclass with
fewer than 3 cells in a given frame triggers a warning (unstable
within-group variance estimate) rather than being silently dropped.

In [ ]:
N_PERMUTATIONS = 999
SEED = 42


def run_modality_stats(modality):
    '''
    Run PERMANOVA + PERMDISP + pairwise PERMANOVA (subclass vs. subclass)
    for every frame in config.FRAMES, for one modality, directly on
    CAJAL's own GW distance matrix (never the UMAP projection).

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        omnibus_df:  one row per frame - n, n_groups, permanova
                     pseudo_F/R2/p, permdisp F/p
        pairwise_df: one row per (frame, subclass pair) - pseudo_F, R2,
                     raw p_value, and q_value (BH-corrected within that
                     frame's 6 pairs)
    '''
    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    selected_path = os.path.join(config.CAJAL_OUTPUT_DIR, 'selected_metadata.csv')
    selected = pd.read_csv(selected_path)

    labels, dist_matrix = load_distance_matrix(dist_csv)
    meta = match_labels_to_metadata(labels, selected)

    omnibus_rows  = []
    pairwise_rows = []
    for frame in config.FRAMES:
        mask = (meta['frame'] == frame).to_numpy()
        sub_dist   = dist_matrix[np.ix_(mask, mask)]
        sub_labels = meta.loc[mask, 'subclass'].to_numpy()

        print(f"[{modality}] frame {frame}: n={mask.sum()}, "
              f"subclasses={pd.Series(sub_labels).value_counts().to_dict()}")

        loc  = permanova(sub_dist, sub_labels, n_permutations=N_PERMUTATIONS, seed=SEED)
        disp = permdisp(sub_dist, sub_labels, n_permutations=N_PERMUTATIONS, seed=SEED)
        omnibus_rows.append({
            'modality': modality, 'frame': frame, 'n': loc['n'], 'n_groups': loc['n_groups'],
            'permanova_pseudo_F': loc['pseudo_F'], 'permanova_R2': loc['R2'], 'permanova_p': loc['p_value'],
            'permdisp_F': disp['F'], 'permdisp_p': disp['p_value'],
        })

        pw = permanova_pairwise(sub_dist, sub_labels, n_permutations=N_PERMUTATIONS, seed=SEED)
        pw.insert(0, 'frame', frame)
        pw.insert(0, 'modality', modality)
        pairwise_rows.append(pw)

    return pd.DataFrame(omnibus_rows), pd.concat(pairwise_rows, ignore_index=True)


## Run: cell morphology

In [ ]:
cell_omnibus, cell_pairwise = run_modality_stats('cell')
cell_omnibus

In [ ]:
cell_pairwise

## Run: nuclear morphology

In [ ]:
nuc_omnibus, nuc_pairwise = run_modality_stats('nuc')
nuc_omnibus

In [ ]:
nuc_pairwise

## Interpreting the results

For each row of `*_omnibus`:
- `permanova_p < 0.05` with high `permanova_R2` and `permdisp_p >= 0.05`:
  a clean location (shape) difference between subclasses in that frame.
- `permanova_p < 0.05` AND `permdisp_p < 0.05`: ambiguous - part or all of
  the PERMANOVA signal may be a dispersion difference, not a location
  difference. Check `permdisp`'s `mean_dispersion` per subclass (printed
  during the run above) to see which subclass is driving it.
- `permanova_p >= 0.05`: no detectable separation in that frame, at this
  sample size - not evidence the subclasses are identical, only that this
  test did not distinguish them.

For `*_pairwise`, use `q_value` (not `p_value`) to decide significance -
it is already corrected for the 6 simultaneous comparisons within each
(modality, frame).

## Save results to disk

In [ ]:
STATS_DIR = os.path.join(config.OUTPUT_DIR, 'stats')
os.makedirs(STATS_DIR, exist_ok=True)

omnibus_df  = pd.concat([cell_omnibus, nuc_omnibus], ignore_index=True)
pairwise_df = pd.concat([cell_pairwise, nuc_pairwise], ignore_index=True)

omnibus_path  = os.path.join(STATS_DIR, 'subclass_separation_omnibus.csv')
pairwise_path = os.path.join(STATS_DIR, 'subclass_separation_pairwise.csv')
omnibus_df.to_csv(omnibus_path, index=False)
pairwise_df.to_csv(pairwise_path, index=False)
print(f"Saved {omnibus_path}")
print(f"Saved {pairwise_path}")